In [ ]:
# install.packages('mgcv')
# install.packages('dplyr')
# install.packages('ggplot2')
# install.packages("gam.hp")
# install.packages('devtools')
# install.packages('visreg')
# install.packages('patchwork')
# install.packages('splineplot')

In [ ]:
library(dplyr)
library(ggplot2)
library(mgcv)
library(visreg)
library(splineplot)
library(patchwork)
library(parallel)
# library(gam.hp)

In [ ]:
full_df = read.csv('data/full_df.csv')

In [ ]:
full_df$ln_area_ha = log(full_df$area_ha)

In [ ]:
full_df$biome <- as.factor(full_df$biome)
full_df$cd_mun <- as.factor(full_df$cd_mun)

In [ ]:
# full_df = full_df[full_df$pop_density<1000,]
# full_df = full_df[full_df$gdp_per_capita<4e+5,]
# full_df = full_df[!is.na(full_df$pop_density),]
# full_df = full_df[!is.na(full_df$gdp_per_capita),]
# full_df = full_df[!is.na(full_df$area_ha),]

In [ ]:
full_df$irrigation_2022_density[is.na(full_df$irrigation_2022)] <- 0
full_df$total_aqua_kilos[is.na(full_df$total_aqua_kilos)] <- 0
full_df$total_aqua_kilos_density[is.na(full_df$total_aqua_kilos_density)] <- 0
# A few cases where rice LULC is less than 0, issue with MB data
full_df$rice_current[full_df$rice_current<0] <- 0
full_df$is_irrigated <- as.factor(full_df$irrigation_2022_density>0)
full_df$natural_total_diff <- full_df$natural_total_diff*-1
full_df$gdp_per_capita[is.na(full_df$gdp_per_capita)] <- mean(full_df$gdp_per_capita, na.rm = TRUE)
full_df[is.na(full_df)] <- 0

In [ ]:
# Best option so far
count_gam = gam(data=full_df, 
family='nb',
link='log',
    count ~ biome + s(cattle_density, k=15) + te(precip_mean, precip_std, k=10) + s(crossing_count_density)
    + s(pop_density, k=5) + s(soy_current, k=5) + s(natural_total_diff, k=5) + s(agriculture_per_capita, k=5) + s(total_aqua_kilos_density)
    + s(rice_quantity_density, k=5) + te(center_lon, center_lat, k=10) + offset(ln_area_ha),
    method="REML")

In [ ]:
summary(count_gam)

In [ ]:
AIC(count_gam)

In [ ]:
concurvity(count_gam, full=TRUE)

In [ ]:

concurvity(count_gam, full=FALSE)

In [ ]:
tiff("/home/ksolvik/research/reservoirs/figs/ch0/gam_check_v2.tif", width = 6.5, height = 6.5, units='in', res=300) 
gam.check(count_gam)
dev.off()

In [ ]:
plot_data <- plot(count_gam, residuals=F, seWithMean = T, all.terms=TRUE, pages=1)

In [ ]:
single_var_list = list(
    cattle_density = expression('Cattle Herd Size (per km'^2*')'),
    pop_density = expression('Population (per km'^2*')'),
    soy_current = 'Soy LULC %',
    natural_total_diff = 'Natural Land-Cover % Lost\n(1985 - 2021)',
    agriculture_per_capita = 'Agr. GDP per capita (1,000 R$)',
    rice_quantity_density = expression('Rice Production (kg/km'^2*')'),
    total_aqua_kilos_density = expression('Aquaculture Production (kg/km'^2*')'),
    crossing_count_density = expression('Road/River Crossings (per km'^2*')'),
    biome = 'Biome'
)



In [ ]:
for (v in names(single_var_list)){
    visreg(count_gam, v, xlab=single_var_list[[v]], ylab='Effect', type='contrast', ylim=c(-2,2),
           xlim=c(min(full_df[[v]]), quantile(full_df[[v]], 0.99)))
}

In [ ]:

zmin=-1
zmax=8
visgam_data <- plot(count_gam, sel=2, too.far=0.1)
visreg_data <- visreg2d(count_gam, "precip_mean", "precip_std", nn=40)
exclude_mask <- matrix(visgam_data[[2]]$exclude, nrow=40, byrow=FALSE)
# Extract visreg surface Vata
z <- visreg_data$z
x <- visreg_data$x
y <- visreg_data$y

# Suppose mask is a logical matrix of same dimension as z
z_masked <- z
z_masked[exclude_mask] <- NA  # mask out unwanted regions

# Plot with ggplot
df_plot <- expand.grid(x = x, y = y)
df_plot$z <- as.vector(z_masked)

precip_plot <- ggplot(df_plot, aes(x, y, fill = z)) +
  geom_raster() +
  scale_fill_viridis_c(option = "plasma", na.value = "transparent", limits = c(zmin, zmax)) +
  xlab('Monthly Mean Precip. (mm)') + ylab('Std. Dev. Precip. (mm)') +
  theme_bw() + 
  theme(panel.grid = element_blank(), legend.position='none')#,legend.position = "none"),#, axis.title.x = element_text(vjust=1)) 
  # theme_bw() + theme(panel.grid = element_blank())

precip_plot
  

In [ ]:
zmin=-1
zmax=8
visgam_data <- plot(count_gam, sel=10, too.far=0.05)
visreg_data <- visreg2d(count_gam, "center_lon", "center_lat", nn=40)
exclude_mask <- matrix(visgam_data[[10]]$exclude, nrow=40, byrow=FALSE)
# Extract visreg surface Vata
z <- visreg_data$z
x <- visreg_data$x
y <- visreg_data$y

# Suppose mask is a logical matrix of same dimension as z
z_masked <- z
z_masked[exclude_mask] <- NA  # mask out unwanted regions

# Plot with ggplot
df_plot <- expand.grid(x = x, y = y)
df_plot$z <- as.vector(z_masked)

latlon_plot <- ggplot(df_plot, aes(x, y, fill = z)) +
  geom_raster() +
  scale_fill_viridis_c(option = "plasma", na.value = "transparent", limits = c(zmin, zmax)) +
  xlab('Lon (deg)') + ylab('Lat (deg)') +
  theme_bw() + 
  theme(panel.grid = element_blank(),legend.position = "none", axis.title.x = element_text(vjust=0.5)) 
  # theme_bw() + theme(panel.grid = element_blank())

latlon_plot
  

In [ ]:
global_options <- list(theme_bw(), theme(panel.grid = element_blank()),
                       theme(axis.text.y = element_text(color = "grey20", size = 10),
                             axis.text.x = element_text(color = "grey20", size=10))
)
# pop_density_options <- list(theme_bw(), theme(panel.grid = element_blank()))
# toprow_shift <- theme(plot.margin=c(0,0,0,0))
options_2d <- list( scale_fill_viridis_c(
  option='plasma',
  limits = c(-1, 8),
  name='Effect'
)
)

In [ ]:
plot_single_var <- function(var_name) {
return(
    visreg(count_gam, var_name, xlab=single_var_list[[var_name]], ylab='Effect', type='contrast',gg=TRUE, 
    line.par = list(col = "#8E3B8E"),points.par = list(cex = 1, alpha=0.2, stroke=0))
    + global_options + theme(axis.text.x = element_text(vjust = 0.5))
    + coord_cartesian(xlim=c(min(full_df[[var_name]]), quantile(full_df[[var_name]], 0.99)), ylim=c(-2,2))
    # + xlim(min(full_df[[var_name]]), quantile(full_df[[var_name]], 0.99))
    )

}

In [ ]:

all_plots <- list(
    plot_single_var('cattle_density'),
    latlon_plot,
    precip_plot + theme(legend.position = "bottom") + labs(fill = "Effect (2D)"),
    plot_single_var('pop_density') ,
    # plot_spacer(),
    guide_area(),
    plot_single_var('crossing_count_density'),
    visreg(count_gam, 'biome', xlab=single_var_list[['biome']], ylab='Effect', type='contrast', gg=TRUE,
        line.par = list(col = "#8E3B8E"), points.par = list(cex = 1, alpha=0.2, stroke=0),
        ) + global_options + theme(axis.text.x = element_text(angle = 30, hjust = 1), axis.title.x=element_text(vjust=1.5))
        + coord_cartesian(ylim=c(-2,2)),
    plot_single_var('natural_total_diff')+ theme(axis.text.x = element_text(vjust = 1),axis.title.x = element_text(vjust = 11)),
    plot_single_var('soy_current')+ theme(axis.text.x = element_text(vjust = 1), axis.title.x = element_text(vjust = 3)),
    plot_single_var('agriculture_per_capita'),
    plot_single_var('total_aqua_kilos_density'),
    plot_single_var('rice_quantity_density')#+ theme(axis.title.x = element_text(vjust=-1))
)

In [ ]:
g <- wrap_plots(all_plots, nrow = 4, ncol = 3) + #, guides = "collect") +
  plot_layout(widths = rep(1, 3), heights = rep(1, 4), guides="collect") 
ggsave("/home/ksolvik/research/reservoirs/figs/ch0/muni_modeling_v3.tif", g, 
       units='in', width = 7.5, height = 10, dpi=400, compression='lzw')

In [ ]:

g <- wrap_plots(all_plots[1:6], nrow = 2, ncol = 3) + #, guides = "collect") +
  plot_layout(widths = rep(1, 3), heights = rep(1, 2)) 
ggsave("/home/ksolvik/research/reservoirs/figs/ch0/muni_modeling_v3_top6.tif", g, 
       units='in', width = 7.5, height = 6, dpi=400, compression='lzw')

In [ ]:

g <- wrap_plots(all_plots[7:11], nrow = 2, ncol = 3) + #, guides = "collect") +
  plot_layout(widths = rep(1, 3), heights = rep(1, 2)) 
ggsave("/home/ksolvik/research/reservoirs/figs/ch0/muni_modeling_v3_last5.tif", g, 
       units='in', width = 7.5, height = 6, dpi=400, compression='lzw')

In [ ]:

fit_subset_model <- function(i, mod_null, dat, iv.name, binarymx, offsetterm, type) {
        tmp.name <- iv.name[as.logical(binarymx[, i])]

        to_add_terms <- tmp.name
        if (!is.null(offsetterm)) to_add_terms <- c(to_add_terms, offsetterm)

        to_add <- paste("~", paste(to_add_terms, collapse = " + "))
        modnew <- stats::update(object = mod_null, data = dat, formula = as.formula(to_add))

        if (type == "dev") {return(summary(modnew)$dev.expl)}
        if (type == "adjR2") {return(summary(modnew)$r.sq)}
}

In [ ]:
# See gam.hp's repo for full package: https://github.com/laijiangshan/gam.hp
# Needed slight modification to allow gam.hp to work with offset

# MAJOR CHANGE: makes sure that te terms are included _together_
get_terms <-
function(mod) {
  raw_names = names(count_gam$sp)
  clean_names = c()
  for (n in raw_names) {
    clean_names = c(clean_names, paste0(strsplit(n, ")")[[1]][1], ')'))
  }
# Add factors
factors <- names(attr(count_gam$terms, 'dataClasses')[attr(count_gam$terms, 'dataClasses')=='factor'])

 return(unique(c(clean_names, factors)))
}

creatbin <-
function(col, binmatrix) {
row<-1
val<-col
	while (val!=0){              
	if (odd(val)) {
		binmatrix[row,col]=1 
	}
	val<-as.integer(val/2)
	row<-row+1
}
##Return matrix
return(binmatrix)
}

genList <-
function(ivlist, value){
numlist  <-  length(ivlist)
newlist <- ivlist
newlist <- 0
for (i in 1:numlist){
	newlist[i] <- abs(ivlist[i])+abs(value)
	if (((ivlist[i]<0) && (value >= 0))|| ((ivlist[i]>=0) && (value <0)))newlist[i]=newlist[i]*-1
}
return(newlist)
}

#'Internal function for glmm.hp() to determine whether the odd number
#' @param  val Imput number.
#' @return a logical value
#' @return \item{Logical value}{TRUE or FALSE}
#' @keywords internal
odd <- function (val) 
{
    if (val%%2 == 0) 
     return(FALSE)
	 else
    return(TRUE)
}
gam.hp <- function(mod, iv = NULL, type = "dev", commonality = FALSE, data = NULL)
{
  # --- 支持 gam 和 bam ---
  if (!(inherits(mod, "gam") || inherits(mod, "bam")))
    stop("gam.hp currently supports 'gam' and 'bam' objects from mgcv package.")

  # --- 检查 RHS 是否含有 '*'（保持原有提示） ---
  rhs_char <- tryCatch(as.character(formula(mod))[3], error = function(e) "")
  if (nzchar(rhs_char) && grepl("\\*", rhs_char, fixed = FALSE))
    stop("Please put the interaction term as a new variable (i.e. link variables by colon(:)) and avoid the asterisk (*) and colon(:) in the original model")

  # --- 更稳健地提取模型项（term labels） ---
  # ivname <- attr(terms(mod), "term.labels")
  ivname <- get_terms(mod)
  if (is.null(ivname)) ivname <- character(0)

  # --- 检查 offset（若有） ---
  offsetterm <- NULL
  offpos <- attr(terms(mod), "offset")
  if (!is.null(offpos) && length(offpos) > 0 && offpos > 0) {
    # 与原来一致的取法
    offsetterm <- attr(mod$terms, "variables")[[offpos + 1]]
  }

  iv.name <- ivname

  # --- 从 summary 中取 R2 / explained deviance ---
  smod <- summary(mod)
  if (type == "adjR2") outr2 <- smod$r.sq else if (type == "dev") outr2 <- smod$dev.expl else stop("type must be 'dev' or 'adjR2'")

  r2type <- row.names(outr2)
  nr2type <- length(r2type)
  if (nr2type == 0) {
    nr2type <- 1
    if (commonality) r2type <- "commonality.analysis" else r2type <- "hierarchical.partitioning"
  }

  # --- 尽量恢复原始数据：优先使用传入 data 参数，其次 mod$model，再尝试 mod$call$data / model.frame --- 
  dat <- NULL
  if (!is.null(data)) {
    dat <- data
  } 
  else {
    if (!is.null(mod$model) && is.data.frame(mod$model)) {
      dat <- mod$model
      names(dat)[names(dat) == "offset(ln_area_ha)"] <- "ln_area_ha"
    } else {
      # try evaluate mod$call$data
      if (!is.null(mod$call$data)) {
        dat_try <- tryCatch(eval(mod$call$data, envir = environment(formula(mod))), error = function(e) NULL)
        if (is.null(dat_try)) dat_try <- tryCatch(eval(mod$call$data, envir = parent.frame()), error = function(e) NULL)
        if (!is.null(dat_try) && is.data.frame(dat_try)) dat <- dat_try
      }
      # fallback to model.frame
      if (is.null(dat)) {
        dat_try2 <- tryCatch(model.frame(formula(mod), data = environment(formula(mod))), error = function(e) NULL)
        if (is.null(dat_try2)) dat_try2 <- tryCatch(model.frame(mod), error = function(e) NULL)
        if (!is.null(dat_try2) && is.data.frame(dat_try2)) dat <- dat_try2
      }
    }
  }

  if (is.null(dat) || !is.data.frame(dat)) {
    stop("Cannot locate the data.frame used to fit the model. Refit the model with a named 'data' argument or call gam.hp(mod, data = your_data).")
  }
  dat <- na.omit(dat)

  # --- 构建 null model (移除所有预测变量) ---
  to_del <- paste(paste("-", iv.name, sep = ""), collapse = " ")
  modnull <- stats::update(stats::formula(mod), paste(". ~ . ", to_del, sep = ""))
  mod_null <- stats::update(object = mod, formula. = modnull, data = dat)

  # --- 以下保持你原来的层次分解算法不变（只用到了 iv.name, dat, mod_null, mod 等） ---
  if (is.null(iv)) {

    nvar <- length(iv.name)
    if (nvar < 2)
      stop("Analysis not conducted. Insufficient number of predictors.")

    totalN <- 2^nvar - 1
    binarymx <- matrix(0, nvar, totalN)
    for (i in 1:totalN) {
      binarymx <- creatbin(i, binarymx)
    }

    outputList <- list()
    outputList[[1]] <- outr2
    for (k in 1:nr2type) {
      commonM <- matrix(nrow = totalN, ncol = 3)

      # HERE"S THE IMPORTANT PART
      explainPower <- mclapply(X = 1:totalN,
                             FUN = fit_subset_model,
                             mc.cores=12,
                             dat=dat,
                             mod_null=mod_null,
                             iv.name=iv.name,
                             binarymx=binarymx,
                             offsetterm=offsetterm,
                             type=type
                             )
      for (i in 1:totalN) {
        commonM[i, 2] <- explainPower[[i]][1]
      }

      commonlist <- vector("list", totalN)

      seqID <- vector()
      for (i in 1:nvar) {
        seqID[i] = 2^(i - 1)
      }

      for (i in 1:totalN) {
        bit <- binarymx[1, i]
        if (bit == 1)
          ivname <- c(0, -seqID[1])
        else ivname <- seqID[1]
        for (j in 2:nvar) {
          bit <- binarymx[j, i]
          if (bit == 1) {
            alist <- ivname
            blist <- genList(ivname, -seqID[j])
            ivname <- c(alist, blist)
          }
          else ivname <- genList(ivname, seqID[j])
        }
        ivname <- ivname * -1
        commonlist[[i]] <- ivname
      }

      for (i in 1:totalN) {
        r2list <- unlist(commonlist[i])
        numlist <- length(r2list)
        ccsum <- 0
        for (j in 1:numlist) {
          indexs <- r2list[[j]]
          indexu <- abs(indexs)
          if (indexu != 0) {
            ccvalue <- commonM[indexu, 2]
            if (indexs < 0)
              ccvalue <- ccvalue * -1
            ccsum <- ccsum + ccvalue
          }
        }
        commonM[i, 3] <- ccsum
      }

      orderList <- vector("list", totalN)
      index <- 0
      for (i in 1:nvar) {
        for (j in 1:totalN) {
          nbits <- sum(binarymx[, j])
          if (nbits == i) {
            index <- index + 1
            commonM[index, 1] <- j
          }
        }
      }

      outputcommonM <- matrix(nrow = totalN + 1, ncol = 2)
      totalRSquare <- sum(commonM[, 3])
      for (i in 1:totalN) {
        outputcommonM[i, 1] <- round(commonM[commonM[i,
                                                     1], 3], digits = 4)
        outputcommonM[i, 2] <- round((commonM[commonM[i,
                                                      1], 3] / totalRSquare) * 100, digits = 2)
      }
      outputcommonM[totalN + 1, 1] <- round(totalRSquare,
                                            digits = 4)
      outputcommonM[totalN + 1, 2] <- round(100, digits = 4)
      rowNames <- NULL
      for (i in 1:totalN) {
        ii <- commonM[i, 1]
        nbits <- sum(binarymx[, ii])
        cbits <- 0
        if (nbits == 1)
          rowName <- "Unique to "
        else rowName <- "Common to "
        for (j in 1:nvar) {
          if (binarymx[j, ii] == 1) {
            if (nbits == 1)
              rowName <- paste(rowName, iv.name[j], sep = "")
            else {
              cbits <- cbits + 1
              if (cbits == nbits) {
                rowName <- paste(rowName, "and ", sep = "")
                rowName <- paste(rowName, iv.name[j], sep = "")
              }
              else {
                rowName <- paste(rowName, iv.name[j], sep = "")
                rowName <- paste(rowName, ", ", sep = "")
              }
            }
          }
        }
        rowNames <- c(rowNames, rowName)
      }
      rowNames <- c(rowNames, "Total")
      rowNames <- format.default(rowNames, justify = "left")
      colNames <- format.default(c("Fractions", " % Total"),
                                 justify = "right")
      dimnames(outputcommonM) <- list(rowNames, colNames)

      VariableImportance <- matrix(nrow = nvar, ncol = 4)
      for (i in 1:nvar) {
        VariableImportance[i, 3] <- round(sum(binarymx[i, ] * (commonM[, 3] / apply(binarymx, 2, sum))), digits = 4)
      }

      VariableImportance[, 1] <- outputcommonM[1:nvar, 1]
      VariableImportance[, 2] <- VariableImportance[, 3] - VariableImportance[, 1]

      total = round(sum(VariableImportance[, 3]), digits = 4)
      VariableImportance[, 4] <- round(100 * VariableImportance[, 3] / total, 2)
      dimnames(VariableImportance) <- list(iv.name, c("Unique", "Average.share", "Individual", "I.perc(%)"))

      if (commonality) {
        outputList[[k + 1]] <- outputcommonM
      } else {
        outputList[[k + 1]] <- VariableImportance
      }
    }
  } else {
    # --- 保留你原来按组 iv 划分的分支（与上面同理，仅用 iv.name 作为变量名来源） ---
    nvar <- length(iv)
    ilist <- names(iv)
    if (is.null(ilist)) {
      names(iv) <- paste("X", 1:nvar, sep = "")
    } else {
      whichnoname <- which(ilist == "")
      if (length(whichnoname) > 0) names(iv)[whichnoname] <- paste("X", whichnoname, sep = "")
    }

    ilist <- names(iv)

    ivlist <- ilist
    iv.name <- ilist

    ivID <- matrix(nrow = nvar, ncol = 1)
    for (i in 0:nvar - 1) {
      ivID[i + 1] <- 2^i
    }

    totalN <- 2^nvar - 1

    binarymx <- matrix(0, nvar, totalN)
    for (i in 1:totalN) {
      binarymx <- creatbin(i, binarymx)
    }

    outputList <- list()
    outputList[[1]] <- outr2
    for (k in 1:nr2type) {
      commonM <- matrix(nrow = totalN, ncol = 3)
      explainPower <- mclapply(X = 1:totalN,
                             FUN = fit_subset_model,
                             mc.cores=detectCores()-1,
                             dat=dat,
                             mod_null=mod_null,
                             iv.name=iv.name,
                             binarymx=binarymx,
                             offsetterm=offsetterm,
                             type=type
                             )
      for (i in 1:totalN) {
        commonM[i, 2] <- explainPower[[i]][1]
      }

      commonlist <- vector("list", totalN)

      seqID <- vector()
      for (i in 1:nvar) {
        seqID[i] = 2^(i - 1)
      }

      for (i in 1:totalN) {
        bit <- binarymx[1, i]
        if (bit == 1)
          ivname <- c(0, -seqID[1])
        else ivname <- seqID[1]
        for (j in 2:nvar) {
          bit <- binarymx[j, i]
          if (bit == 1) {
            alist <- ivname
            blist <- genList(ivname, -seqID[j])
            ivname <- c(alist, blist)
          }
          else ivname <- genList(ivname, seqID[j])
        }
        ivname <- ivname * -1
        commonlist[[i]] <- ivname
      }

      for (i in 1:totalN) {
        r2list <- unlist(commonlist[i])
        numlist <- length(r2list)
        ccsum <- 0
        for (j in 1:numlist) {
          indexs <- r2list[[j]]
          indexu <- abs(indexs)
          if (indexu != 0) {
            ccvalue <- commonM[indexu, 2]
            if (indexs < 0)
              ccvalue <- ccvalue * -1
            ccsum <- ccsum + ccvalue
          }
        }
        commonM[i, 3] <- ccsum
      }

      orderList <- vector("list", totalN)
      index <- 0
      for (i in 1:nvar) {
        for (j in 1:totalN) {
          nbits <- sum(binarymx[, j])
          if (nbits == i) {
            index <- index + 1
            commonM[index, 1] <- j
          }
        }
      }

      outputcommonM <- matrix(nrow = totalN + 1, ncol = 2)
      totalRSquare <- sum(commonM[, 3])
      for (i in 1:totalN) {
        outputcommonM[i, 1] <- round(commonM[commonM[i,
                                                     1], 3], digits = 4)
        outputcommonM[i, 2] <- round((commonM[commonM[i,
                                                      1], 3] / totalRSquare) * 100, digits = 2)
      }
      outputcommonM[totalN + 1, 1] <- round(totalRSquare,
                                            digits = 4)
      outputcommonM[totalN + 1, 2] <- round(100, digits = 4)
      rowNames <- NULL
      for (i in 1:totalN) {
        ii <- commonM[i, 1]
        nbits <- sum(binarymx[, ii])
        cbits <- 0
        if (nbits == 1)
          rowName <- "Unique to "
        else rowName <- "Common to "
        for (j in 1:nvar) {
          if (binarymx[j, ii] == 1) {
            if (nbits == 1)
              rowName <- paste(rowName, iv.name[j], sep = "")
            else {
              cbits <- cbits + 1
              if (cbits == nbits) {
                rowName <- paste(rowName, "and ", sep = "")
                rowName <- paste(rowName, iv.name[j], sep = "")
              }
              else {
                rowName <- paste(rowName, iv.name[j], sep = "")
                rowName <- paste(rowName, ", ", sep = "")
              }
            }
          }
        }
        rowNames <- c(rowNames, rowName)
      }
      rowNames <- c(rowNames, "Total")
      rowNames <- format.default(rowNames, justify = "left")
      colNames <- format.default(c("Fractions", " % Total"),
                                 justify = "right")
      dimnames(outputcommonM) <- list(rowNames, colNames)

      VariableImportance <- matrix(nrow = nvar, ncol = 4)
      for (i in 1:nvar) {
        VariableImportance[i, 3] <- round(sum(binarymx[i, ] * (commonM[, 3] / apply(binarymx, 2, sum))), digits = 4)
      }

      VariableImportance[, 1] <- outputcommonM[1:nvar, 1]
      VariableImportance[, 2] <- VariableImportance[, 3] - VariableImportance[, 1]

      total = round(sum(VariableImportance[, 3]), digits = 4)
      VariableImportance[, 4] <- round(100 * VariableImportance[, 3] / total, 2)
      dimnames(VariableImportance) <- list(iv.name, c("Unique", "Average.share", "Individual", "I.perc(%)"))

      if (commonality) {
        outputList[[k + 1]] <- outputcommonM
      } else {
        outputList[[k + 1]] <- VariableImportance
      }
    }
  }

  if (type == "adjR2") {
    names(outputList) <- c("adjusted.R2", r2type)
  }
  if (type == "dev") {
    names(outputList) <- c("Explained.deviance", r2type)
  }
  outputList$variables <- iv.name
  if (commonality) {
    outputList$type <- "commonality.analysis"
  }
  if (!commonality) {
    outputList$type <- "hierarchical.partitioning"
  }
  class(outputList) <- "gamhp"
  outputList
}


In [ ]:
detectCores()

In [ ]:

hp_results = gam.hp(count_gam)

In [ ]:
hp_results

In [ ]:
hp_results$hierarchical.partitioning